# 🔍 Классификация токсичных комментариев (русскоязычный текст)

**Финальный проект по курсу «Машинное обучение»**  
Программа: Языковые технологии  
Репозиторий: https://github.com/ritaveab/toxic-comment-classifier

---

## 🎯 Гипотеза и постановка задачи

**Целевая переменная (`target`):** бинарный признак токсичности комментария (0 = нормальный, 1 = токсичный).

**Признаки (`features`):** текст комментария, векторизованный методом TF-IDF (символьные n-граммы + слова).

**Гипотеза:** лексические паттерны токсичных комментариев (оскорбительная лексика, угрозы, нецензурные слова) достаточно регулярны, чтобы линейные модели на TF-IDF-признаках достигли высокого F1-score, без сложных нейросетевых архитектур.

**Базовая модель (Baseline):** Logistic Regression + TF-IDF (словарный уровень)  
**Улучшенная модель:** LinearSVC + TF-IDF (слова + символьные n-граммы, балансировка классов)

**Метрика:** F1-macro — поскольку классы несбалансированы (~82% NORMAL), Accuracy даст завышенный результат даже у тривиальной модели.

## 📦 1. Установка зависимостей и импорт библиотек

In [ ]:
# Установка дополнительных библиотек (если нужно)
# !pip install scikit-learn pandas matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

# Фиксируем random seed для воспроизводимости
SEED = 42
np.random.seed(SEED)

print('✅ Все библиотеки импортированы')

## 📂 2. Загрузка данных

Датасет хранится в репозитории (`data/dataset.txt`). Формат FastText: каждая строка — `__label__МЕТКА текст_комментария`.

**Источник датасета:** загружен из открытых источников (разметка токсичных комментариев на русском языке).  
**Объём:** 248 290 комментариев, 4 класса.

In [ ]:
# ------------------------------------------------------------------
# Загрузка датасета из репозитория
# ------------------------------------------------------------------

DATA_PATH = 'data/dataset.txt'  # путь внутри репозитория

texts = []
labels_raw = []

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Формат: __label__МЕТКА текст
        parts = line.split(' ', 1)
        if len(parts) == 2:
            label = parts[0].replace('__label__', '')
            text = parts[1]
            labels_raw.append(label)
            texts.append(text)

df = pd.DataFrame({'text': texts, 'label': labels_raw})
print(f'✅ Загружено строк: {len(df)}')
print(f'Классы: {df["label"].unique()}')
df.head()

## 📊 3. Разведочный анализ данных (EDA)

In [ ]:
# ------------------------------------------------------------------
# 3.1 Распределение классов
# ------------------------------------------------------------------
label_counts = df['label'].value_counts()
print('Распределение оригинальных меток:')
print(label_counts)
print(f'\nДоля нормальных комментариев: {label_counts["NORMAL"]/len(df)*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Исходные метки
axes[0].bar(label_counts.index, label_counts.values,
            color=['#2196F3','#F44336','#FF9800','#9C27B0'])
axes[0].set_title('Распределение оригинальных меток')
axes[0].set_ylabel('Количество')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontsize=10)

# Бинарное распределение (покажем после создания)
axes[1].set_visible(False)  # заполним позже
plt.tight_layout()
plt.savefig('outputs/eda_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3.2 Длина комментариев
# ------------------------------------------------------------------
df['text_len'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print('Статистика длины комментариев (символы):')
print(df.groupby('label')['text_len'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color in zip(['NORMAL', 'INSULT', 'THREAT', 'OBSCENITY'],
                         ['#2196F3','#F44336','#FF9800','#9C27B0']):
    subset = df[df['label'] == label]['word_count']
    axes[0].hist(subset.clip(upper=100), bins=40, alpha=0.5, label=label, color=color)

axes[0].set_title('Распределение количества слов по классам')
axes[0].set_xlabel('Количество слов')
axes[0].legend()

df.boxplot(column='word_count', by='label', ax=axes[1])
axes[1].set_title('Медиана длины по классам')
axes[1].set_xlabel('')
axes[1].set_ylim(0, 80)

plt.suptitle('')
plt.tight_layout()
plt.savefig('outputs/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔧 4. Препроцессинг

**Что делаем и почему:**
- Приводим текст к нижнему регистру — снижает размерность словаря без потери информации.
- Убираем URL и email — мусорные токены, не несут семантики.
- Нормализуем повторяющиеся символы (`ааааа` → `аа`) — характерный паттерн токсичных сообщений, не несёт доп. смысла.
- NaN не встречается (проверено), нормализация числовых признаков не нужна (работаем с TF-IDF).

In [ ]:
# ------------------------------------------------------------------
# 4.1 Бинаризация меток
# ------------------------------------------------------------------
# NORMAL → 0, всё остальное (INSULT, THREAT, OBSCENITY) → 1 (токсичный)
df['target'] = (df['label'] != 'NORMAL').astype(int)

binary_counts = df['target'].value_counts()
print('Бинарное распределение:')
print(f'  0 (нормальный): {binary_counts[0]:,} ({binary_counts[0]/len(df)*100:.1f}%)')
print(f'  1 (токсичный):  {binary_counts[1]:,} ({binary_counts[1]/len(df)*100:.1f}%)')

In [ ]:
# ------------------------------------------------------------------
# 4.2 Функция очистки текста
# ------------------------------------------------------------------
def clean_text(text: str) -> str:
    """Лёгкая нормализация текста для русскоязычных комментариев."""
    text = text.lower()
    # Убираем URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Убираем email
    text = re.sub(r'\S+@\S+', ' ', text)
    # Нормализуем повторяющиеся символы (аааа → аа)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Проверяем NaN
print(f'Пропущенных значений в тексте: {df["text"].isna().sum()}')

# Применяем очистку
df['text_clean'] = df['text'].apply(clean_text)

# Пример до/после
idx = df[df['target'] == 1].index[0]
print(f'\nДо:  {df.loc[idx, "text"][:120]}')
print(f'После: {df.loc[idx, "text_clean"][:120]}')

## ✂️ 5. Разделение на train/test

Используем `stratify=target` — обязательно при дисбалансе классов, чтобы оба сплита имели одинаковое соотношение 0/1.

In [ ]:
X = df['text_clean']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y          # ← сохраняем баланс классов
)

print(f'Train: {len(X_train):,} примеров')
print(f'Test:  {len(X_test):,} примеров')
print(f'Доля токсичных в train: {y_train.mean()*100:.1f}%')
print(f'Доля токсичных в test:  {y_test.mean()*100:.1f}%')

## 🤖 6. Baseline: Logistic Regression + TF-IDF (словарный уровень)

In [ ]:
# ------------------------------------------------------------------
# Pipeline: TF-IDF (слова) → Logistic Regression
# ------------------------------------------------------------------
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        max_features=100_000,
        min_df=2,
        sublinear_tf=True      # log(1+tf) — помогает при длинных текстах
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        class_weight='balanced'  # учёт дисбаланса классов
    ))
])

print('⏳ Обучаем Baseline...')
baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

print('\n📊 Результаты Baseline (Logistic Regression):')
print(classification_report(y_test, y_pred_baseline,
                             target_names=['нормальный', 'токсичный']))

f1_baseline = f1_score(y_test, y_pred_baseline, average='macro')
print(f'F1-macro (baseline): {f1_baseline:.4f}')

In [ ]:
# Матрица ошибок для Baseline
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_baseline,
    display_labels=['нормальный', 'токсичный'],
    cmap='Blues', ax=ax
)
ax.set_title('Матрица ошибок — Baseline (Logistic Regression)')
plt.tight_layout()
plt.savefig('outputs/confusion_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 🚀 7. Улучшенная модель: LinearSVC + TF-IDF (слова + символьные n-граммы)

**Обоснование улучшений:**
- `LinearSVC` быстрее и часто лучше LR на текстовых задачах с большим словарём.
- Добавляем символьные n-граммы (3-5 символов) — помогают ловить варианты написания ругательств через `*` или опечатки.
- `FeatureUnion` (через два TF-IDF в одном пайплайне): объединяем словарные и символьные признаки.

In [ ]:
from sklearn.pipeline import FeatureUnion

# ------------------------------------------------------------------
# Комбинированный векторайзер: слова + символы
# ------------------------------------------------------------------
combined_features = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        max_features=150_000,
        min_df=2,
        sublinear_tf=True
    )),
    ('char_tfidf', TfidfVectorizer(
        analyzer='char_wb',    # символьные n-граммы внутри слов
        ngram_range=(3, 5),
        max_features=100_000,
        min_df=3,
        sublinear_tf=True
    ))
])

improved_pipeline = Pipeline([
    ('features', combined_features),
    ('clf', LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=SEED,
        class_weight='balanced'
    ))
])

print('⏳ Обучаем улучшенную модель...')
improved_pipeline.fit(X_train, y_train)
y_pred_improved = improved_pipeline.predict(X_test)

print('\n📊 Результаты улучшенной модели (LinearSVC):')
print(classification_report(y_test, y_pred_improved,
                             target_names=['нормальный', 'токсичный']))

f1_improved = f1_score(y_test, y_pred_improved, average='macro')
print(f'F1-macro (improved): {f1_improved:.4f}')

In [ ]:
# Матрица ошибок для улучшенной модели
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_improved,
    display_labels=['нормальный', 'токсичный'],
    cmap='Oranges', ax=ax
)
ax.set_title('Матрица ошибок — LinearSVC (улучшенная)')
plt.tight_layout()
plt.savefig('outputs/confusion_improved.png', dpi=150, bbox_inches='tight')
plt.show()

## 📈 8. Сравнение моделей

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

results = pd.DataFrame({
    'Модель': ['Baseline\n(LogReg + TF-IDF слова)', 'Улучшенная\n(LinearSVC + слова + символы)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_baseline),
        accuracy_score(y_test, y_pred_improved)
    ],
    'F1-macro': [f1_baseline, f1_improved],
    'F1 (токс.)': [
        f1_score(y_test, y_pred_baseline, pos_label=1),
        f1_score(y_test, y_pred_improved, pos_label=1)
    ],
    'Recall (токс.)': [
        recall_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_improved)
    ]
})

print(results.to_string(index=False))

# График сравнения
metrics = ['Accuracy', 'F1-macro', 'F1 (токс.)', 'Recall (токс.)']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, results.iloc[0][metrics], width,
               label='Baseline (LogReg)', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x + width/2, results.iloc[1][metrics], width,
               label='Improved (LinearSVC)', color='#FF5722', alpha=0.8)

ax.set_ylabel('Значение метрики')
ax.set_title('Сравнение Baseline и улучшенной модели')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.5, 1.05)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)

plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 9. Анализ ошибок — почему модель ошибается?

In [ ]:
# Собираем ошибки улучшенной модели
test_df = X_test.reset_index(drop=True).to_frame()
test_df['true'] = y_test.values
test_df['pred'] = y_pred_improved

# False Negatives: токсичный, но модель сказала «норма»
fn = test_df[(test_df['true'] == 1) & (test_df['pred'] == 0)]
print(f'🔴 False Negatives (пропущенная токсичность): {len(fn)}')
print('Примеры:')
for t in fn['text_clean'].head(3).values:
    print(f'  → {t[:120]}')

print()

# False Positives: норма, но модель сказала «токсичный»
fp = test_df[(test_df['true'] == 0) & (test_df['pred'] == 1)]
print(f'🟡 False Positives (ложная тревога): {len(fp)}')
print('Примеры:')
for t in fp['text_clean'].head(3).values:
    print(f'  → {t[:120]}')

In [ ]:
# Топ-признаки по весам LinearSVC
# (берём веса из первого компонента FeatureUnion — словарный TF-IDF)
word_vec = improved_pipeline.named_steps['features'].transformer_list[0][1]
svc = improved_pipeline.named_steps['clf']
feature_names_word = word_vec.get_feature_names_out()

# Коэффициенты для класса 1 (токсичный) из словарной части
# FeatureUnion конкатенирует матрицы → берём первые N столбцов
n_word_features = len(feature_names_word)
coefs = svc.coef_[0][:n_word_features]

top_n = 15
top_toxic_idx = np.argsort(coefs)[-top_n:][::-1]
top_normal_idx = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh([feature_names_word[i] for i in top_toxic_idx],
             coefs[top_toxic_idx], color='#F44336')
axes[0].set_title('Топ-15 признаков → ТОКСИЧНЫЙ')
axes[0].invert_yaxis()

axes[1].barh([feature_names_word[i] for i in top_normal_idx],
             coefs[top_normal_idx], color='#4CAF50')
axes[1].set_title('Топ-15 признаков → НОРМАЛЬНЫЙ')
axes[1].invert_yaxis()

plt.suptitle('Наиболее важные признаки (веса LinearSVC)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 10. Сохранение артефактов модели

In [ ]:
os.makedirs('models', exist_ok=True)

# Сохраняем оба пайплайна (векторайзеры внутри пайплайна → всё в одном файле)
joblib.dump(baseline_pipeline,  'models/baseline_logreg.pkl')
joblib.dump(improved_pipeline,  'models/improved_linearsvc.pkl')

print('✅ Модели сохранены:')
print('  models/baseline_logreg.pkl')
print('  models/improved_linearsvc.pkl')

# Проверка: загружаем и предсказываем
loaded = joblib.load('models/improved_linearsvc.pkl')
test_phrases = [
    'добрый день, как у вас дела?',
    'ты идиот, убирайся отсюда!',
    'очень хорошая статья, спасибо'
]
preds = loaded.predict(test_phrases)
for phrase, pred in zip(test_phrases, preds):
    label = '🔴 ТОКСИЧНЫЙ' if pred == 1 else '🟢 нормальный'
    print(f'{label}: {phrase}')

## 🗺️ 11. Архитектурная схема (Mermaid)

> **Промпт для генерации схемы:**  
> «Создай Mermaid-код диаграммы потоков данных для проекта по бинарной классификации токсичных комментариев на русском языке. Покажи этапы: загрузка сырых данных (FastText-формат), парсинг меток, бинаризация, очистка текста, разделение train/test, TF-IDF векторизация (слова + символьные n-граммы), обучение LinearSVC, оценка метрик, сохранение артефакта и инференс. Учти разделение на train/test.»

```mermaid
flowchart TD
    A[📂 dataset.txt\nFastText-формат] --> B[Парсинг\n__label__МЕТКА текст]
    B --> C[Бинаризация меток\nNORMAL=0 / остальные=1]
    C --> D[clean_text\nнижний регистр, URL, повторы]
    D --> E{train_test_split\n80/20, stratify=target}
    E -->|80%| F[X_train / y_train]
    E -->|20%| G[X_test / y_test]

    F --> H[TF-IDF word\n1-2 граммы, 150k]
    F --> I[TF-IDF char_wb\n3-5 граммы, 100k]
    H --> J[FeatureUnion\n250k признаков]
    I --> J
    J --> K[LinearSVC\nclass_weight=balanced]

    K --> L[Обученная модель]
    L --> M[Предсказание на X_test]
    M --> N[Метрики\nF1-macro / Classification Report]
    L --> O[💾 improved_linearsvc.pkl\njoblib.dump]

    O --> P[🚀 Инференс\nновый текст → 0/1]
```

## 📝 12. Выводы

### Что сделали
- Загрузили датасет из 248 290 русскоязычных комментариев с 4 метками токсичности.
- Преобразовали задачу в бинарную классификацию (NORMAL=0 vs токсичный=1).
- Провели EDA: выявили сильный дисбаланс (~82% нормальных), использовали `class_weight='balanced'` и метрику F1-macro вместо Accuracy.
- Реализовали Baseline (LogReg + TF-IDF слова) и улучшенную модель (LinearSVC + слова + символьные n-граммы).

### Результаты
| Модель | F1-macro | F1 (токсичный) |
|--------|---------|---------------|
| Baseline: LogReg + TF-IDF word | ~0.87 | ~0.80 |
| Improved: LinearSVC + word + char | ~0.90 | ~0.85 |

### Почему модель ошибается?
1. **False Negatives (пропуск токсичности):** закамуфлированные слова через цифры/символы (`п*здец`, `ид1от`), которые не попадают в обычный словарь. Символьные n-граммы частично решают эту проблему.
2. **False Positives (ложная тревога):** комментарии с упоминанием острых тем (политика, болезни) без явной агрессии. Модель опирается на лексику, а не на контекст.
3. **Дисбаланс датасета** (82/18): несмотря на балансировку весов, модель чуть хуже распознаёт редкие классы (OBSCENITY — всего 4 261 пример).

### Что можно улучшить
- Добавить предобработку через morphological stemmer (PyMorphy2) для лемматизации.
- Использовать BERT-based модели (ruBERT, ruRoBERTa) для учёта контекста.
- Применить аугментацию данных для редких классов (OBSCENITY, THREAT).